In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, shutil
import numpy as np, pandas as pd

ROOT     = "/content/drive/MyDrive/dtn_project"
RES_A    = f"{ROOT}/results"
RES_B    = f"{ROOT}/results_b"
LOCAL    = "/content/processed"
WINDOW   = 12
DAY      = 288                          # operator: 288 five-minute steps = 1 day
DATASETS = ["5g", "4g", "operator"]
COLSET   = {"5g": "single", "4g": "single", "operator": "value+daily"}
os.makedirs(RES_B, exist_ok=True)
if not os.path.exists(LOCAL):
    shutil.copytree(f"{ROOT}/processed", LOCAL)
for ds in DATASETS:
    print(ds, len(glob.glob(f"{LOCAL}/{ds}/*.npz")), "traces")
print("Outputs will be saved in:", RES_B)

Mounted at /content/drive
5g 79 traces
4g 134 traces
operator 55 traces
Outputs will be saved in: /content/drive/MyDrive/dtn_project/results_b


## 2. Data loading and metrics
- `load(ds)` reads every trace (train / val / test arrays) and the scaler. The **first column is always the forecast target**.
- For every test step we need the true value, the previous true value, and the value one day earlier. All come from the full series (train + val + test joined), which uses only values that were already known at that time.


In [3]:
def load(ds):
    d = f"{LOCAL}/{ds}"
    sc = json.load(open(f"{d}/scaler.json"))
    out = {}
    for f in sorted(glob.glob(d + "/*.npz")):
        z = np.load(f)
        out[os.path.basename(f)[:-4]] = (z["train"][:, 0], z["val"][:, 0], z["test"][:, 0])
    return out, sc["mean"][0], sc["std"][0]

def metrics(y, p, last):
    y, p, last = (np.asarray(a, dtype="float64") for a in (y, p, last))
    err = p - y
    ss = ((y - y.mean()) ** 2).sum()
    ch = y - last; m = ch != 0
    return dict(MAE=np.abs(err).mean(), RMSE=np.sqrt((err ** 2).mean()),
                R2=1 - (err ** 2).sum() / ss if ss > 0 else np.nan,
                dir_acc=(np.sign(p - last)[m] == np.sign(ch)[m]).mean() if m.any() else np.nan)

def test_arrays(tr, va, te):
    """True values of the test period, the previous true value, and the value one day earlier."""
    full = np.concatenate([tr, va, te]); idx = len(tr) + len(va) + np.arange(len(te))
    y, last = full[idx], full[idx - 1]
    day_ago = full[idx - DAY] if idx[0] - DAY >= 0 else None
    return y, last, day_ago

## 3. Run the baselines
Scores each baseline on every test trace and saves two files in `results_b/`
- `baseline_trace_results.csv`: one row per trace (MAE, RMSE, R², direction accuracy, `run_id`). These feed the statistics tests.
- `baseline_runs.csv`: one row per baseline.

Baselines have no training and no seeds, so `seed` is 0 and `train_s` is 0. Their speed is measured later, together with all other models, in the cost notebook.

In [4]:
rows_all, runs = [], []
for ds in DATASETS:
    traces, mu0, sd0 = load(ds)
    kinds = ["last_value"] + (["seasonal_naive"] if ds == "operator" else [])
    for kind in kinds:
        rid = f"{ds}|{COLSET[ds]}|w{WINDOW}|{kind}|s0"
        rows = []
        for k, (tr, va, te) in traces.items():
            y, last, day = test_arrays(tr, va, te)
            p = last if kind == "last_value" else day
            if p is None:
                print("skipped (series shorter than a day):", k); continue
            rows.append(dict(trace=k, n=len(y), run_id=rid,
                             **metrics(y * sd0 + mu0, p * sd0 + mu0, last * sd0 + mu0)))
        rows_all += rows
        runs.append(dict(run_id=rid, dataset=ds, colset=COLSET[ds], window=WINDOW, model=kind, seed=0,
                         hidden=0, lr=0, params=0, epochs=0, best_val=np.nan, train_s=0.0))
        print(f"{rid}: mean RMSE {np.mean([r['RMSE'] for r in rows]):.3f} | "
              f"median R2 {np.nanmedian([r['R2'] for r in rows]):.2f} | {len(rows)} traces")
pd.DataFrame(rows_all).to_csv(f"{RES_B}/baseline_trace_results.csv", index=False)
pd.DataFrame(runs).to_csv(f"{RES_B}/baseline_runs.csv", index=False)
print("saved to", RES_B)

5g|single|w12|last_value|s0: mean RMSE 9352.828 | median R2 -0.14 | 79 traces
4g|single|w12|last_value|s0: mean RMSE 3429.096 | median R2 0.59 | 134 traces
operator|value+daily|w12|last_value|s0: mean RMSE 15.437 | median R2 0.97 | 55 traces
operator|value+daily|w12|seasonal_naive|s0: mean RMSE 49.903 | median R2 0.56 | 55 traces
saved to /content/drive/MyDrive/dtn_project/results_b


In [5]:
na = pd.read_csv(f"{RES_A}/neural_trace_results.csv")
na["model"] = na.run_id.str.split("|").str[3]; na["dataset"] = na.run_id.str.split("|").str[0]
na["colset"] = na.run_id.str.split("|").str[1]; na["window"] = na.run_id.str.split("|").str[2]
nb = pd.DataFrame(rows_all)
nb["model"] = nb.run_id.str.split("|").str[3]; nb["dataset"] = nb.run_id.str.split("|").str[0]

for ds in DATASETS:
    a = na[(na.dataset == ds) & (na.colset == COLSET[ds]) & (na.window == f"w{WINDOW}")]
    b = nb[nb.dataset == ds]
    print(f"{ds}: traces neural {a.trace.nunique()} | baselines {b.trace.nunique()} | same set: {set(a.trace) == set(b.trace)}")

both = pd.concat([na[(na.colset == na.dataset.map(COLSET)) & (na.window == f"w{WINDOW}")], nb])
tab = both.groupby(["dataset", "model"]).agg(RMSE=("RMSE", "mean"), MAE=("MAE", "mean"),
                                              R2_median=("R2", "median"), dir_acc=("dir_acc", "mean")).round(3)
print(); print(tab.to_string())

5g: traces neural 79 | baselines 79 | same set: True
4g: traces neural 134 | baselines 134 | same set: True
operator: traces neural 55 | baselines 55 | same set: True

                             RMSE       MAE  R2_median  dir_acc
dataset  model                                                 
4g       bilstm          3294.491  2248.928      0.615    0.579
         conv1d          3277.317  2235.066      0.632    0.586
         gru             3279.344  2232.588      0.621    0.580
         last_value      3429.096  2268.447      0.593    0.000
         lstm            3286.527  2244.676      0.614    0.577
5g       bilstm          7843.485  4310.465      0.090    0.624
         conv1d          8096.221  4727.087      0.018    0.606
         gru             7900.200  4509.970      0.029    0.624
         last_value      9352.828  4572.245     -0.136    0.000
         lstm            7868.564  4472.215      0.089    0.630
operator bilstm            15.772     9.023      0.943    0.563
